# 0. Import

In [1]:
# ────────────────────────────────────────────────
# 0. 라이브러리 및 설정
# ────────────────────────────────────────────────
import os
import time
import gc
import torch
import numpy as np
import pandas as pd
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
from torch_geometric.utils import negative_sampling
from torch.amp import GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from neo4j.exceptions import ServiceUnavailable
from neomodel import config as neomodel_config, db

In [2]:
# 하드웨어 설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cuda


In [3]:
def cypher_df(q: str, params=None) -> pd.DataFrame:
    """
    Cypher 쿼리를 실행하여 pandas DataFrame으로 반환.
    모든 *_id, mid, oid, pid 컬럼을 정수형으로 캐스팅.
    """
    rec, cols = db.cypher_query(q, params or {})
    df = pd.DataFrame(rec, columns=cols)
    id_cols = [c for c in df.columns if c.endswith('_id') or c in {'mid', 'oid', 'pid'}]
    for col in id_cols:
        df[col] = (
            pd.to_numeric(df[col], errors='coerce')
              .astype('Int64')      # Nullable Int
              .astype('int64')      # 확정 int64
        )
    return df

def safe_map(series: pd.Series, mapping: dict, col: str) -> np.ndarray:
    """
    series의 각 값을 mapping 딕셔너리로 매핑. 실패한 값은 drop.
    반환: int64 NumPy 배열
    """
    mapped = series.map(mapping)
    miss = mapped.isna().sum()
    if miss:
        print(f"⚠️  {col}: {miss:,} unmapped → drop")
        mapped = mapped.dropna()
    return mapped.astype(np.int64).to_numpy()

### Connnect Neo4j

In [4]:
# Neo4j 연결 정보 (사용자 환경에 맞게 수정)
from app.config.local import (NEO4J_HOST, NEO4J_PORT, NEO4J_USER, DB_PASSWORD)
neomodel_config.DATABASE_URL = (
    f"bolt://{NEO4J_USER}:{DB_PASSWORD}@{NEO4J_HOST}:{NEO4J_PORT}"
)
try:
    result, _ = db.cypher_query("RETURN 1 AS test")
    print("✅ Neo4j 연결 성공! → 응답값:", result[0][0])
except ServiceUnavailable as e:
    print("❌ Neo4j 연결 실패:", e)

✅ config_module: ['CARD_SERVER_ADDRESS', 'DB_HOST', 'DB_NAME', 'DB_PASSWORD', 'DB_PORT', 'DB_USER', 'DEBUG', 'GEMINI_API_KEY', 'GOOGLE_APPLICATION_CREDENTIALS', 'GOOGLE_PROJECT', 'NEO4J_HOST', 'NEO4J_PORT', 'NEO4J_USER', 'SWAGGER_SPECS_JSON_ROUTE', 'SWAGGER_SPECS_ROUTE', 'SWAGGER_STATIC_PATH', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'os']
✅ Neo4j 연결 성공! → 응답값: 1


### Load Model, index

In [5]:
# HetSAGE 모델 (dropout 적용)
class HetSAGE(nn.Module):
    def __init__(self, hid=128, out=128, p_drop=0.2, aggr='sum'):
        super().__init__()
        self.conv1 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((-1, -1), hid, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((-1, -1), hid, aggr=aggr),
        }, aggr=aggr)
        self.conv2 = HeteroConv({
            ('user', 'ordered', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'contains', 'product'): SAGEConv((hid, hid), out, aggr=aggr),
            ('order', 'ordered_by', 'user'): SAGEConv((hid, hid), out, aggr=aggr),
            ('product', 'contained_in', 'order'): SAGEConv((hid, hid), out, aggr=aggr),
        }, aggr=aggr)
        self.norm1 = nn.LayerNorm(hid)
        self.norm2 = nn.LayerNorm(out)
        self.p_drop = p_drop

    def _drop(self, h: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        return {k: F.dropout(v, p=self.p_drop, training=self.training) for k, v in h.items()}

    def forward(self, x_dict, edge_index_dict):
        h = self.conv1(x_dict, edge_index_dict)
        h = {k: F.relu(self.norm1(v)) for k, v in h.items()}
        h = self._drop(h)
        h = self.conv2(h, edge_index_dict)
        h = {k: F.relu(self.norm2(v)) for k, v in h.items()}
        h = self._drop(h)
        return h

In [6]:
ckpt_path = "checkpoints/best_model.pt"  # 실제 파일명 확인 (확장자 필수!)

In [7]:
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
hid_dim = ckpt.get('hid_dim', 128)
out_dim = ckpt.get('out_dim', 128)
model = HetSAGE(hid=hid_dim, out=out_dim)
model.load_state_dict(ckpt['model_state'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()

# 5. 인덱스 매핑 복원
m2i = ckpt['m2i']
o2i = ckpt['o2i']
p2i = ckpt['p2i']
print(f"🔗 인덱스 매핑 복원 (user: {len(m2i)}, order: {len(o2i)}, product: {len(p2i)})")

🔗 인덱스 매핑 복원 (user: 206209, order: 3346083, product: 49685)


### Load Data

In [8]:
# ────────────────────────────────────────────────
# 3. Order & Product Features 로드
# ────────────────────────────────────────────────
# 3-1. Order feature
df_order_feat = cypher_df("""
MATCH (o:Order)
RETURN o.order_id AS oid,
       o.days_since_prior_order_norm AS dsp,
       o.order_dow_norm              AS dow,
       o.order_hour_of_day_norm      AS hr,
       o.order_number_norm           AS idx
""")
df_order_feat["oid_idx"] = safe_map(df_order_feat.oid, o2i, "oid")
order_feat = (
    df_order_feat
      .sort_values("oid_idx")[["dsp","dow","hr","idx"]]
      .to_numpy(np.float32)
)
print(f"Order feature shape: {order_feat.shape}")

# 3-2. Product feature (category_vector + name_vector)
df_prod_vec = cypher_df("""
MATCH (p:Product)
RETURN p.product_id AS pid,
       p.category_vector + p.name_vector AS vec
""")
# product_id가 문자열이므로 safe_map 시 str(pid) 사용
df_prod_vec["pid_idx"] = safe_map(df_prod_vec.pid.map(str), p2i, "pid")
prod_feat = torch.tensor(
    np.stack(df_prod_vec
              .sort_values("pid_idx")
              .vec.values),
    dtype=torch.float32
)
print(f"Product feature shape: {prod_feat.shape}")

# ────────────────────────────────────────────────
# 4. 테스트용 간선 및 gt_dict 생성
# ────────────────────────────────────────────────
# 4-1. prior(테스트 입력) 간선: eval_set='test' AND role_test='prior'
df_edge_te = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='prior'
RETURN m.member_id AS mid, o.order_id AS oid, p.product_id AS pid
""")
uo_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.mid, m2i, "mid")),
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid"))
), 0).to(torch.long)
op_te = torch.stack((
    torch.from_numpy(safe_map(df_edge_te.oid, o2i, "oid")),
    # product_id를 str로 매핑
    torch.from_numpy(safe_map(df_edge_te.pid.map(str), p2i, "pid"))
), 0).to(torch.long)
print(f"Test-prior edges: uo_te={uo_te.size(1):,}, op_te={op_te.size(1):,}")

# 4-2. ground-truth: eval_set='test' AND role_test='train'
df_gt = cypher_df("""
MATCH (m:Member)-[:ORDERED]->(o:Order)-[:CONTAINS]->(p:Product)
WHERE o.eval_set='test' AND o.role_test='train'
RETURN m.member_id AS mid, collect(p.product_id) AS gt_list
""")
mid_arr = safe_map(df_gt.mid, m2i, "mid")
gt_list_arr = [
    [p2i[str(pid)] for pid in pid_list if str(pid) in p2i]
    for pid_list in df_gt.gt_list
]
gt_dict = {
    int(mid): [int(pid) for pid in pid_list]
    for mid, pid_list in zip(mid_arr, gt_list_arr)
    if mid != -1
}
print(f"GT users (index): {len(gt_dict):,}")

# ────────────────────────────────────────────────
# 5. 부분 그래프 생성 함수 (patch 적용)
# ────────────────────────────────────────────────
def build_subgraph_for_users(
    user_ids: np.ndarray,
    uo_te: torch.Tensor,
    op_te: torch.Tensor,
    order_feat: np.ndarray,
    prod_feat: torch.Tensor,
    m2i: dict[int,int],
    o2i: dict[int,int],
    p2i: dict[str,int],
    gt_dict: dict[int, list[int]]
) -> tuple[HeteroData, dict[int, list[int]]]:
    """
    ▸ user_ids : 전역 user 인덱스 배열 (예: [12, 53, 102, ...])
    ▸ uo_te, op_te : [2, E] 형태의 prior 간선 (유저→오더, 오더→프로덕트)
    ▸ order_feat, prod_feat : 전체 오더/프로덕트 피처
    ▸ m2i, o2i, p2i : 전역 ID → 인덱스 맵
    ▸ gt_dict : {전역 유저인덱스: [전역 프로덕트인덱스, ...], ...}
    반환
      ▸ 부분 HeteroData (유저, 오더, 프로덕트 서브셋)
      ▸ 로컬 인덱스로 변환된 gt_local : {로컬_user_idx: [로컬_prod_idx, ...], ...}
    """
    # 1) user_ids의 전역 인덱스 배열 → user_idx (전역 → 인덱스)
    user_idx = np.array([m2i[u] for u in user_ids], dtype=np.int64)

    # 2) prior 유저→오더 간선 필터링
    mask_uo = np.isin(uo_te[0].cpu().numpy(), user_idx)
    uo_sub = uo_te[:, mask_uo]
    order_idx = np.unique(uo_sub[1].cpu().numpy())

    # 3) prior 오더→프로덕트 간선 필터링
    mask_op = np.isin(op_te[0].cpu().numpy(), order_idx)
    op_sub = op_te[:, mask_op]
    product_idx = np.unique(op_sub[1].cpu().numpy())

    # 4) 전역 → 로컬 인덱스 매핑
    old2new_user  = {old: new for new, old in enumerate(user_idx)}
    old2new_order = {old: new for new, old in enumerate(order_idx)}
    old2new_prod  = {old: new for new, old in enumerate(product_idx)}

    # 5) edge_index 리매핑
    uo_remap = torch.stack([
        torch.tensor([old2new_user[u.item()]  for u in uo_sub[0]]),
        torch.tensor([old2new_order[o.item()] for o in uo_sub[1]])
    ], 0)
    op_remap = torch.stack([
        torch.tensor([old2new_order[o.item()] for o in op_sub[0]]),
        torch.tensor([old2new_prod[p.item()]  for p in op_sub[1]])
    ], 0)

    # 6) 노드 피처 추출 (로컬 인덱스 순서대로)
    order_x   = torch.tensor(order_feat[order_idx], dtype=torch.float32)
    product_x = prod_feat[product_idx]

    # 7) 로컬 ground-truth 생성 (전역 → 로컬 매핑)
    gt_local = {
        old2new_user[u]:
        [old2new_prod[p] for p in gt_dict[u] if p in old2new_prod]
        for u in user_ids if u in gt_dict
    }

    # 8) HeteroData 객체 구성
    d = HeteroData()
    d['user'].num_nodes    = len(user_idx)
    d['order'].num_nodes   = len(order_idx)
    d['product'].num_nodes = len(product_idx)
    d['user'].x    = torch.nn.Embedding(len(user_idx), 32).weight
    d['order'].x   = order_x
    d['product'].x = product_x
    d['user','ordered','order'].edge_index         = uo_remap
    d['order','ordered_by','user'].edge_index      = uo_remap.flip(0)
    d['order','contains','product'].edge_index     = op_remap
    d['product','contained_in','order'].edge_index = op_remap.flip(0)

    return d, gt_local

# ────────────────────────────────────────────────
# 6. 검증 함수 (메모리 관리 포함)
# ────────────────────────────────────────────────
@torch.no_grad()
def validate_minibatch(
    model: nn.Module,
    user_ids: np.ndarray,
    uo_te: torch.Tensor,
    op_te: torch.Tensor,
    order_feat: np.ndarray,
    prod_feat: torch.Tensor,
    m2i: dict[int,int],
    o2i: dict[int,int],
    p2i: dict[str,int],
    gt_dict: dict[int, list[int]],
    TOP_K: int = 5,
    SUB_SIZE: int = 10_000,
    CHUNK: int = 200_000
) -> float:
    """
    ▸ 모든 user_ids를 SUB_SIZE 단위로 나눠 GPU forward (OOM 방지)
    ▸ Precision@TOP_K 계산 후 전체 평균 반환
    """
    model.eval()
    precs: list[float] = []

    # 1) SUB_SIZE 단위로 유저 나누기
    for s in range(0, len(user_ids), SUB_SIZE):
        batch_users = user_ids[s:s + SUB_SIZE]

        # (1) 부분 그래프 & 로컬 gt 생성
        g_sub, gt_sub = build_subgraph_for_users(
            batch_users, uo_te, op_te,
            order_feat, prod_feat,
            m2i, o2i, p2i, gt_dict
        )

        # (2) GPU forward
        g_sub = g_sub.to(device, non_blocking=True)
        z = model(g_sub.x_dict, g_sub.edge_index_dict)
        uemb = z['user'].cpu()      # [len(batch_users), hidden_dim]
        pemb = z['product'].cpu()   # [len(sub_prod), hidden_dim]

        # (3) 각 로컬 유저 별 Precision@K
        for uid_local, gt_local in gt_sub.items():
            u = uemb[uid_local]
            best_score, best_idx = [], []
            for i in range(0, len(pemb), CHUNK):
                sc, ix = (pemb[i:i + CHUNK] @ u).topk(TOP_K)
                best_score.append(sc);  best_idx.append(ix + i)
            best_score = torch.cat(best_score)
            best_idx   = torch.cat(best_idx)

            top_pred = best_idx[best_score.topk(TOP_K).indices].tolist()
            hit = len(set(top_pred) & set(gt_local))
            precs.append(hit / TOP_K)

        # (4) 메모리 반환
        del g_sub, z, uemb, pemb
        torch.cuda.empty_cache()
        gc.collect()

    return float(np.mean(precs)) if precs else 0.0

⚠️  oid: 11 unmapped → drop


ValueError: Length of values (3346083) does not match length of index (3346094)

# Evaluate

In [ ]:
# ────────────────────────────────────────────────
# 8. 전체 테스트 유저에 대해 Precision@TOP_K 계산
# ────────────────────────────────────────────────
all_test_users = np.array(list(gt_dict.keys()), dtype=np.int64)
print(f"전체 테스트 유저 수: {len(all_test_users):,}")

start_time = time.perf_counter()
overall_prec = validate_minibatch(
    model,
    all_test_users,
    uo_te, op_te,
    order_feat, prod_feat,
    m2i, o2i, p2i,
    gt_dict,
    TOP_K=5,
    SUB_SIZE=10_000,    # 10k 단위로 나눠 OOM 방지
    CHUNK=200_000
)
elapsed = time.perf_counter() - start_time

print(f"\n🎯 전체 Precision@5: {overall_prec:.4f}")
print(f"⏱  평가 소요 시간: {elapsed/60:.1f} 분")